# 4. Time series: does a tax exemption raise fertility? (FERTIL3)

**Question:** what is the long-run effect of the personal tax exemption (pe) on the U.S. general
fertility rate (gfr), 1913–1984? Effects plausibly take time → distributed lag.
(Wooldridge Examples 10.4 and 11.8.)

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo

from _setup import *  # noqa: F401,F403
import diagnostics as dg
import specification as sp

## Steps 1–3 — Expected sign positive (a subsidy to children). Annual series, 72 years.

In [2]:
fz = woo.data("fertil3")
fz.index = pd.Index(fz["year"].astype(int), name="year")
for k in (1, 2):
    fz[f"pe_{k}"] = fz["pe"].shift(k)

## Step 5 first: are the series stationary?
Regressing trending series in levels risks a spurious regression.

In [3]:
st = dg.stationarity_many(fz[["gfr", "pe"]])


Stationarity — gfr (N=72): I(1)
     Transform Deterministics     ADF  p ADF  lags  DF-GLS  p DF-GLS   KPSS  p KPSS                                         Verdict
         level       constant -2.3064 0.1699     6 -0.8563    0.3542 0.4160  0.0703 Inconclusive (high persistence or long memory?)
         level constant+trend -2.8391 0.1830     6 -2.2989    0.1767 0.1558  0.0418                                       Unit root
1st difference       constant -2.6706 0.0793     3 -5.6512    0.0000 0.1993  0.1000                                      Stationary
1st difference constant+trend -2.6490 0.2578     3 -6.0335    0.0000 0.1981  0.0167                         Conflict (trend/break?)

Stationarity — pe (N=72): I(1)
     Transform Deterministics     ADF  p ADF  lags  DF-GLS  p DF-GLS   KPSS  p KPSS    Verdict
         level       constant -1.8714 0.3456     1 -1.1216    0.2457 0.7903  0.0100  Unit root
         level constant+trend -1.4713 0.8388     1 -1.6125    0.5155 0.2446  0.0100  

Both series look integrated of order one, so the levels regression of Example 10.4 is at risk of
being spurious. We show it, then estimate in first differences (Example 11.8).

In [4]:
lev = smf.ols("gfr ~ pe + pe_1 + pe_2 + ww2 + pill", fz).fit()
lr_lev = sp.long_run_effect(lev, ["pe", "pe_1", "pe_2"], show=True)
_ = dg.ols_diagnostics(lev, time_series=True)


long-run propensity (Σβ): 0.1007 (se 0.0298), 95% CI [0.0412, 0.1603], p = 0.0012

OLS diagnostics (N=70, R²=0.499, adj. R²=0.459, condition number=1314.9)
                     Test                                       H0  Statistic  p-value                                                                                                                                      Conclusion
    Ramsey RESET (ŷ², ŷ³)                  correct functional form     5.5248   0.0062                                                                                Likely misspecification: respecify BEFORE fixing standard errors
            Breusch-Pagan                         homoskedasticity    18.7280   0.0022                                                                                                       Heteroskedasticity: use HC3 (or WLS/FGLS)
             White (full)                         homoskedasticity    36.8335   0.0035                                                                

R² above Durbin-Watson and strong residual autocorrelation: the warning signs of a spurious
regression. Now in first differences, with HAC errors.

In [5]:
for c in ["gfr", "pe", "pe_1", "pe_2"]:
    fz[f"d_{c}"] = fz[c].diff()
nw = dg.newey_west_lags(len(fz))
dif = smf.ols("d_gfr ~ d_pe + d_pe_1 + d_pe_2", fz).fit(cov_type="HAC", cov_kwds={"maxlags": nw})
print(dif.summary().tables[1])
lr_dif = sp.long_run_effect(dif, ["d_pe", "d_pe_1", "d_pe_2"], show=True)
_ = dg.ols_diagnostics(dif, time_series=True)

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.9637      0.576     -1.673      0.094      -2.093       0.165
d_pe          -0.0362      0.036     -1.013      0.311      -0.106       0.034
d_pe_1        -0.0140      0.036     -0.389      0.697      -0.084       0.056
d_pe_2         0.1100      0.022      5.082      0.000       0.068       0.152

long-run propensity (Σβ): 0.0598 (se 0.0517), 95% CI [-0.0415, 0.1611], p = 0.2471

OLS diagnostics (N=69, R²=0.232, adj. R²=0.197, condition number=20.7)
                     Test                                       H0  Statistic  p-value                      Conclusion
    Ramsey RESET (ŷ², ŷ³)                  correct functional form     1.3258   0.2729 No evidence of misspecification
            Breusch-Pagan                         homoskedasticity     1.5111   0.6797                     No evidence
             

## Step 7 — Report
In levels, the long-run propensity is about 0.10 births per 1,000 women per dollar of exemption,
but the regression shows the signature of spuriousness. In first differences, the contemporaneous
and one-year effects are slightly negative and indistinguishable from zero; only the two-year lag is
clearly positive. The long-run effect
is estimated imprecisely, with a confidence interval that includes zero. Conclusion: the evidence
for a fertility response is fragile and depends on treating trending series correctly. Individual
lag coefficients are imprecise because the lags are collinear, so report the long-run effect and
its interval.